# Session 2 — Iteration 7: Class-Based RAG Architecture

Refactors the ingestion pipeline into a clean `VectorDB` class inspired by the
[Anthropic Contextual Embeddings recipe](https://platform.claude.com/cookbook/capabilities-contextual-embeddings-guide).

**Architecture pattern:**
- `VectorDB.__init__` — wire up ChromaDB (persistent), OpenAI embedding fn, and text splitter
- `VectorDB.load_data()` — load PDFs, chunk them, skip if collection already populated
- `VectorDB._embed_and_store()` — batch-embed with OpenAI and upsert into ChromaDB
- `VectorDB.search()` — query ChromaDB, return structured results

**Key change from v6:** ChromaDB `PersistentClient` replaces `EphemeralClient`,
so the index survives kernel restarts (analogous to the recipe's pickle save/load).

In [ ]:
import csv
import os
import re
import tiktoken
import chromadb
import openai as openai_client
import anthropic
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv(find_dotenv())

# Model names — change here to update everywhere
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL       = "claude-sonnet-4-5"

In [ ]:
# Regex to find section headings inside a text chunk (used for citation metadata)
_HEADING_RE = re.compile(
    r'(?:Chapter\s+[A-Z]?\d+[^\n]*|ARTICLE\s+[IVXLC]+[^\n]*|§\s*[A-Z]?\d+[-\w]*\.[^\n]*)',
    re.MULTILINE,
)

def _extract_heading(text: str, page: int) -> str:
    """Return the first section header found in the chunk, or fall back to page number."""
    match = _HEADING_RE.search(text)
    if match:
        return match.group(0).strip()[:80]
    return f"p. {page + 1}"

In [ ]:
class VectorDB:
    """
    ChromaDB-backed vector store for municipal regulation PDFs.

    Mirrors the interface of the Anthropic VectorDB recipe but uses:
      - ChromaDB PersistentClient instead of Voyage AI + pickle
      - OpenAI text-embedding-3-small for embeddings
      - LangChain PyPDFLoader + RecursiveCharacterTextSplitter for ingestion
    """

    # ------------------------------------------------------------------ #
    #  Construction                                                        #
    # ------------------------------------------------------------------ #

    def __init__(
        self,
        name: str,
        data_dir: Path,
        api_key: str | None = None,
        chroma_path: str = "./data/chroma_store",
    ):
        """
        Args:
            name:        ChromaDB collection name — also used as a unique key on disk.
            data_dir:    Directory containing the PDF files to index.
            api_key:     OpenAI API key (falls back to OPENAI_API_KEY env var).
            chroma_path: Folder where ChromaDB stores its on-disk index.
        """
        if api_key is None:
            api_key = os.environ["OPENAI_API_KEY"]

        self.name     = name
        self.data_dir = data_dir
        self._api_key = api_key

        # PersistentClient means the index survives kernel restarts
        self._chroma     = chromadb.PersistentClient(path=chroma_path)
        self._embed_fn   = OpenAIEmbeddingFunction(
            api_key=api_key, model_name=EMBEDDING_MODEL
        )
        self._collection = self._chroma.get_or_create_collection(
            name=name, embedding_function=self._embed_fn
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""],
        )
        self._enc = tiktoken.get_encoding("cl100k_base")

        print(f"VectorDB '{name}' ready — {self._collection.count():,} chunks on disk")

    # ------------------------------------------------------------------ #
    #  Public API                                                          #
    # ------------------------------------------------------------------ #

    def load_data(self) -> None:
        """
        Load every PDF from data_dir into the collection.

        Idempotent: skips ingestion when the collection already contains chunks,
        mirroring the recipe's 'already loaded' guard.
        """
        if self._collection.count() > 0:
            print(
                f"Collection '{self.name}' already loaded "
                f"({self._collection.count():,} chunks). Skipping ingestion."
            )
            return

        # 1. Load raw pages via LangChain
        raw_docs   = []
        pdf_paths  = sorted(self.data_dir.glob("*.pdf"))
        for pdf_path in pdf_paths:
            raw_docs.extend(PyPDFLoader(str(pdf_path)).load())
        print(f"Loaded {len(raw_docs)} pages from {len(pdf_paths)} PDFs")

        # 2. Split into fixed-size chunks
        chunks = self._splitter.split_documents(raw_docs)

        # 3. Normalise into a list of dicts (text + citation metadata)
        sections = [
            {
                "text":        doc.page_content,
                "source":      Path(doc.metadata.get("source", "")).name,
                "chunk_index": i,
                "heading":     _extract_heading(
                    doc.page_content, doc.metadata.get("page", 0)
                ),
            }
            for i, doc in enumerate(chunks)
        ]
        print(
            f"Split into {len(sections):,} chunks "
            f"(avg {sum(len(s['text']) for s in sections) // len(sections)} chars/chunk)"
        )

        self._embed_and_store(sections)

    def search(self, query: str, n_results: int = 7) -> list[dict]:
        """
        Embed query with OpenAI and return top-n chunks.

        Returns:
            List of dicts with keys: text, metadata, score (0–1, higher = more similar).
        """
        results = self._collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )
        return [
            {
                "text":     text,
                "metadata": meta,
                "score":    round(1 - dist, 4),
            }
            for text, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _embed_and_store(
        self,
        sections: list[dict],
        batch_size: int = 500,
        max_tokens: int = 7_500,
    ) -> None:
        """
        Batch-embed texts with the OpenAI API and upsert them into ChromaDB.

        Args:
            sections:   List of dicts (text, source, chunk_index, heading).
            batch_size: Max documents per OpenAI embeddings request.
            max_tokens: Token ceiling per document (OpenAI limit is 8,191).
        """
        oa = openai_client.OpenAI(api_key=self._api_key)

        # Sanitize and truncate every chunk up front
        texts = [self._sanitize(s["text"], max_tokens) for s in sections]

        # Embed in controlled batches (OpenAI caps at 2,048 inputs per request)
        embeddings: list[list[float]] = []
        for i in range(0, len(texts), batch_size):
            batch    = texts[i : i + batch_size]
            response = oa.embeddings.create(input=batch, model=EMBEDDING_MODEL)
            embeddings.extend([r.embedding for r in response.data])
            print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")

        # Upsert into ChromaDB
        self._collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=[
                {
                    "source":      s["source"],
                    "chunk_index": s["chunk_index"],
                    "heading":     s["heading"],
                }
                for s in sections
            ],
            ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in sections],
        )
        print(f"Stored {len(sections):,} chunks in collection '{self.name}'")

    def _sanitize(self, text: str, max_tokens: int | None = None) -> str:
        """Strip null bytes and truncate to max_tokens if provided."""
        text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
        if max_tokens:
            tokens = self._enc.encode(text)
            if len(tokens) > max_tokens:
                text = self._enc.decode(tokens[:max_tokens])
        return text

In [ ]:
# Instantiate and load — subsequent runs skip ingestion because ChromaDB persists to disk.
db = VectorDB(
    name="baldwin_openai",
    data_dir=Path("data/baldwin"),
)
db.load_data()

## Interactive RAG Chat

Uses `db.search()` as the retrieval layer and Anthropic Claude as the LLM.

**Session commands:**
| Command | Effect |
|---------|--------|
| `quit` / `q` | End the session |
| `clear` | Wipe conversation memory and reset token counter |

In [ ]:
# Governance config (override via .env)
TOKEN_BUDGET = int(os.environ.get("RAG_TOKEN_BUDGET", 50_000))
TIMEOUT_SECS = int(os.environ.get("RAG_TIMEOUT_SECS", 30))
LOG_FILE     = Path(os.environ.get("RAG_LOG_FILE", "rag_session_log.csv"))

# Session state (persists across re-runs of this cell)
conversation_history: list[dict] = []
session_tokens_used: int = 0


def _log_turn(query: str, hits: list[dict], answer: str, tokens: int) -> None:
    """Append one row per query to the CSV log file."""
    is_new = not LOG_FILE.exists()
    n = len(hits)
    chunk_headers = [
        f"chunk_{i+1}_{f}" for i in range(n) for f in ("score", "source", "heading", "text")
    ]
    fieldnames = ["timestamp", "query", "answer", "tokens_turn"] + chunk_headers

    row = {
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "query":       query,
        "answer":      answer,
        "tokens_turn": tokens,
    }
    for i, hit in enumerate(hits):
        p = f"chunk_{i+1}_"
        row[p + "score"]   = hit["score"]
        row[p + "source"]  = hit["metadata"]["source"]
        row[p + "heading"] = hit["metadata"]["heading"]
        row[p + "text"]    = hit["text"].replace("\n", " ")

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow(row)
    print(f"[Logged to {LOG_FILE}]")


def compare_interactive(n_results: int = 7) -> None:
    """
    Interactive RAG chat session.

    Retrieval is handled by `db.search()`; answers come from Claude.
    Session memory grows turn-by-turn and is sent as prior context.
    """
    global conversation_history, session_tokens_used

    ac = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    print(f"\nRAG Chat  |  collection: '{db.name}'  |  model: {LLM_MODEL}")
    print(f"Governance: token budget={TOKEN_BUDGET:,}  |  timeout={TIMEOUT_SECS}s")
    print("Commands: 'quit' to exit  |  'clear' to reset memory\n")

    while True:
        remaining = TOKEN_BUDGET - session_tokens_used
        if remaining <= 0:
            print(
                f"[Session token budget of {TOKEN_BUDGET:,} exhausted. "
                "Type 'clear' to reset or 'quit' to exit.]"
            )

        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("quit", "exit", "q"):
            print(f"Session ended. Tokens used: {session_tokens_used:,}")
            break
        if query.lower() == "clear":
            conversation_history.clear()
            session_tokens_used = 0
            print("[Memory cleared. Token counter reset.]\n")
            continue
        if session_tokens_used >= TOKEN_BUDGET:
            print("[Cannot process — token budget exhausted. Type 'clear' to reset.]\n")
            continue

        # Retrieve relevant chunks via the VectorDB class
        hits = db.search(query, n_results=n_results)

        context_str = "\n\n".join(
            f"[{i+1}] {h['metadata']['source']} | {h['metadata']['heading']}\n{h['text']}"
            for i, h in enumerate(hits)
        )

        # Build message list from growing conversation history
        messages = []
        for turn in conversation_history:
            messages.append({"role": "user",      "content": turn["question"]})
            messages.append({"role": "assistant",  "content": turn["answer"]})
        messages.append({"role": "user", "content": query})

        system_prompt = (
            "You are a precise assistant answering questions about Baldwin Borough "
            "municipal regulations. You must follow these rules strictly:\n\n"
            "1. Answer ONLY from the numbered context passages below. "
            "Do NOT use your training data or general knowledge.\n"
            "2. Every factual claim MUST be followed immediately by an inline "
            "citation in the form [N] referencing the passage number it came from.\n"
            "3. If a question cannot be answered from the context, say exactly: "
            "'This information is not available in the provided documents.'\n"
            "4. Never speculate or infer beyond what the passages explicitly state.\n\n"
            f"CONTEXT PASSAGES:\n{context_str}"
        )

        # Call Claude with a hard timeout
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    ac.messages.create,
                    model=LLM_MODEL,
                    max_tokens=1024,
                    system=system_prompt,
                    messages=messages,
                )
                response = future.result(timeout=TIMEOUT_SECS)
        except FuturesTimeoutError:
            print(f"\n[Request timed out after {TIMEOUT_SECS}s. Please try again.]\n")
            continue
        except Exception as e:
            print(f"\n[API error: {e}]\n")
            continue

        turn_tokens      = response.usage.input_tokens + response.usage.output_tokens
        session_tokens_used += turn_tokens
        remaining_after  = TOKEN_BUDGET - session_tokens_used
        answer           = response.content[0].text

        print(f"\n{'─' * 60}")
        print(f"You asked: {query}")
        print(f"{'─' * 60}")
        print(f"\nAssistant: {answer}")

        print("\n" + "─" * 60)
        print("Retrieved passages (cited as [N] above):")
        for i, hit in enumerate(hits):
            print(
                f"\n  [{i+1}] {hit['metadata']['source']} | "
                f"{hit['metadata']['heading']}  (score: {hit['score']})"
            )
            print(f"  {hit['text']}")
        print("\n" + "─" * 60)
        print(
            f"[Tokens this turn: {turn_tokens:,} | "
            f"Session total: {session_tokens_used:,} | "
            f"Budget remaining: {remaining_after:,}]\n"
        )

        conversation_history.append({"question": query, "answer": answer})
        _log_turn(query, hits, answer, turn_tokens)


compare_interactive()